In [ ]:
# =========================
# RQ1: Baseline Model Performance
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=50000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Prepare Features and Target
# -------------------------
X = pd.get_dummies(df.drop(TARGET, axis=1), drop_first=True).astype(float)

le = LabelEncoder()
y = le.fit_transform(df[TARGET])
y = (y > 0).astype(int)

# -------------------------
# 3. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 4. Scaling
# -------------------------
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

X_train_scaled = X_train
X_test_scaled = X_test

# -------------------------
# 5. Baseline Models
# -------------------------
# models = {
#     "Logistic Regression": LogisticRegression(
#         max_iter=1000,
#         random_state=42
#     ),
#     "Decision Tree": DecisionTreeClassifier(
#         random_state=42
#     ),
#     "k-NN": KNeighborsClassifier(
#         n_neighbors=5
#     )
# }

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        solver="saga",
        random_state=42,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "k-NN": KNeighborsClassifier(
        n_neighbors=5
    )
}
# -------------------------
# 6. Train and Evaluate
# -------------------------
results = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

# -------------------------
# 7. Save Table
# -------------------------
table = pd.DataFrame(results)
table.to_csv("RQ1_table.csv", index=False)

print("\n=== RQ1 Baseline Model Performance Table ===")
print(table)

# -------------------------
# 8. Grouped Bar Chart
# -------------------------
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]

x = np.arange(len(table["Model"]))
width = 0.18

plt.figure(figsize=(9, 6))

for i, metric in enumerate(metrics):
    values = table[metric]

    bars = plt.bar(
        x + (i - 1.5) * width,
        values,
        width,
        label=metric
    )

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("RQ1: Baseline Model Performance", fontsize=14, fontweight="bold")
plt.xlabel("Model", fontsize=12, fontweight="bold")
plt.ylabel("Score", fontsize=12, fontweight="bold")

plt.xticks(x, table["Model"])
plt.ylim(0, 1.05)

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("RQ1_figure.pdf", bbox_inches="tight")
plt.show()